# Chapter 6: Bornhuetter–Ferguson and Benktander Methods

Code of Chapter 6 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 6.2.1 Worked example

In [ ]:
import numpy as np
import pandas as pd
from gibf import bornhuetter_ferguson, benktander, credibility_weight
from giexpected import expected_ultimate
from gidevelop import average_factors, cumulative_factors, develop

T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=pd.Index([2022, 2023, 2024, 2025], name="accident_year"),
                 columns=pd.Index([12, 24, 36, 48], name="age"))
EP = pd.Series([2500, 2700, 2900, 3100.0], index=T.index)
dev = develop(T, cumulative_factors(average_factors(T, "volume"), 1.0))
eu = expected_ultimate(EP, 0.70)
bf = bornhuetter_ferguson(dev["latest"], dev["cdf"], eu)
bf["bf_ultimate"].sum().round(2)                   # 7770.41
benktander(dev["latest"], dev["cdf"], eu, k=2).round(2).tolist()
# [1700.0, 1906.05, 1972.8, 2192.87]

## 6.2.2 Iterating to the development method

In [ ]:
for k in (0, 1, 2, 3, 10):
    print(k, benktander(dev["latest"], dev["cdf"], eu, k).loc[[2024, 2025]].round(2).tolist())

## 6.3.1 The synthetic portfolio

In [ ]:
from gitriangle import synthetic_history, triangle, exposure_by_accident_year
from giexpected import trended_ratios

pol, cl, txs = synthetic_history(2020, 2025, seed=100)
I = triangle(cl, txs, "incurred", 2021, 2025)
P = triangle(cl, txs, "paid", 2021, 2025)
E = exposure_by_accident_year(pol, range(2021, 2026))
dI = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))
dP = develop(P, cumulative_factors(average_factors(P, "volume"), 1.0))
mature = dI["ultimate"].loc[2021:2023]
pp = pd.Series({y: trended_ratios(mature, E["earned_exposure"].loc[2021:2023], 0.04, y).mean()
                for y in range(2021, 2026)})
eu_s = E["earned_exposure"] * pp
bI = bornhuetter_ferguson(dI["latest"], dI["cdf"], eu_s)
bP = bornhuetter_ferguson(dP["latest"], dP["cdf"], eu_s)

## 6.3.2 Results

In [ ]:
bI["bf_ultimate"].sum().round(0), bP["bf_ultimate"].sum().round(0)   # 12923259, 12691411
benktander(dI["latest"], dI["cdf"], eu_s, 2).sum().round(0)           # 12949786

## 6.4.1 Sensitivity to a random shock

In [ ]:
shocked = dI["latest"].copy()
shocked[2025] *= 1.10
(bornhuetter_ferguson(shocked, dI["cdf"], eu_s)["bf_ultimate"] - bI["bf_ultimate"])[2025]   # 283874.11

## Exercise 1: Benktander by hand

In [ ]:
credibility_weight(dev["cdf"], k=2).round(4).tolist()   # [1.0, 0.9991, 0.9873, 0.8332]

## Exercise 2: Sensitivity of BF and Benktander to the ELR

In [ ]:
for elr in (0.60, 0.80):
    e = expected_ultimate(EP, elr)
    print(elr, bornhuetter_ferguson(dev["latest"], dev["cdf"], e)["bf_ultimate"][2025].round(2),
          benktander(dev["latest"], dev["cdf"], e, 2)[2025].round(2))
# 0.6 2059.63 2141.16 / 0.8 2312.84 2244.57

## 6.7 Python Walkthrough

In [ ]:
from gibf import bornhuetter_ferguson, benktander, credibility_weight

# Section 6.2: the example triangle with a 70% ELR
bf = bornhuetter_ferguson(dev["latest"], dev["cdf"], eu)
bf[["pct_unreported", "bf_ibnr", "bf_ultimate"]].round(2)
benktander(dev["latest"], dev["cdf"], eu, k=2).sum().round(2)      # 7771.71
credibility_weight(dev["cdf"], k=1).round(4).tolist()               # [1.0, 0.9706, 0.8874, 0.5916]

# Section 6.3: the synthetic portfolio, paid and incurred
benktander(dP["latest"], dP["cdf"], eu_s, 2).round(0).tolist()
# [2324983, 2105313, 2384199, 2874034, 3043414]